# 🚀 Zyro Dynamics HR Help Desk — RAG Challenge
**Optimized RAG Pipeline | BGE Embeddings | Multi-Query | MMR | LangSmith Tracing**

In [ ]:
# Cell 1 — Install dependencies
!pip install -q langchain langchain-community langchain-text-splitters \
    langchain-huggingface langchain-groq langchain-core langsmith \
    faiss-cpu pypdf sentence-transformers huggingface_hub

In [ ]:
# Cell 2 — Imports & API Keys
import os
import re
import base64
import pandas as pd
from kaggle_secrets import UserSecretsClient

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Load secrets from Kaggle
secrets = UserSecretsClient()
os.environ["GROQ_API_KEY"]      = secrets.get_secret("GROQ_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = secrets.get_secret("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"]    = "zyro-rag-challenge"

print("✅ API keys loaded")

In [ ]:
# Cell 3 — TODO: Load HR PDFs from dataset
import glob

# Kaggle dataset path (adjust if needed)
DOCS_PATH = "/kaggle/input/zyro-dynamics-hr-policies"
# Fallback: search for PDFs anywhere
pdf_files = glob.glob(f"{DOCS_PATH}/**/*.pdf", recursive=True)
if not pdf_files:
    pdf_files = glob.glob("/kaggle/input/**/*.pdf", recursive=True)

print(f"📄 Found {len(pdf_files)} PDF files:")
for f in sorted(pdf_files):
    print(f"   {os.path.basename(f)}")

# Load all PDFs
loader    = PyPDFDirectoryLoader(DOCS_PATH if os.path.exists(DOCS_PATH) else os.path.dirname(pdf_files[0]))
documents = loader.load()
print(f"\n✅ Loaded {len(documents)} pages from {len(pdf_files)} documents")

In [ ]:
# Cell 4 — TODO: Chunk documents
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", "! ", "? ", "; ", ", ", " ", ""],
    add_start_index=True,
)
chunks = splitter.split_documents(documents)
print(f"✅ Created {len(chunks)} chunks (size=700, overlap=200)")

In [ ]:
# Cell 5 — TODO: Build FAISS vector store with BGE embeddings
print("⏳ Loading BGE-base embedding model (may take ~60s on first run)...")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore    = FAISS.from_documents(chunks, embeddings)
base_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 6, "fetch_k": 20, "lambda_mult": 0.6},
)
print("✅ FAISS vector store built with MMR retrieval")

In [ ]:
# Cell 6 — TODO: Build LCEL RAG chain with guardrails

# LLM
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_tokens=1024)

# ── Prompts ────────────────────────────────────────────────────────────────────
rag_prompt = ChatPromptTemplate.from_template("""\
You are the official HR Help Desk assistant for Zyro Dynamics Pvt. Ltd.
Answer employee HR questions STRICTLY using the provided policy documents.

CRITICAL RULES:
1. Use ONLY information from the Context — never invent or add external knowledge.
2. Be SPECIFIC: include exact numbers, durations, percentages, eligibility criteria, and procedures.
3. If context clearly has the answer, give it completely and precisely.
4. If context does NOT contain sufficient information, say exactly:
   "This specific information is not covered in the available HR policy documents. Please contact HR directly."
5. Write in clear, professional English. Use bullet points for multi-part answers.

Context (from Zyro Dynamics HR Policy Documents):
{context}

Employee Question: {question}

HR Assistant Answer:"""
)

scope_prompt = ChatPromptTemplate.from_template("""\
You are a classifier. Is the following question related to company HR policies,
employment, leave, benefits, compensation, workplace conduct, IT security,
onboarding, travel expenses, or performance management at a company?

Answer ONLY with one word: YES or NO.

Question: {question}
Answer:"""
)

multi_query_prompt = ChatPromptTemplate.from_template("""\
Generate 3 different phrasings of the following HR question to improve document retrieval.
Output ONLY the 3 questions, one per line, no numbering, no extra text.

Original question: {question}

3 rephrased questions:"""
)

# ── Helper functions ───────────────────────────────────────────────────────────
HR_KEYWORDS = {
    "leave", "vacation", "pto", "sick", "maternity", "paternity", "wfh",
    "remote", "hybrid", "office", "salary", "ctc", "bonus", "increment",
    "appraisal", "performance", "pip", "review", "rating", "probation",
    "onboarding", "offboarding", "separation", "resignation", "notice",
    "travel", "expense", "reimbursement", "policy", "handbook", "conduct",
    "harassment", "posh", "icc", "device", "security", "data",
    "confidential", "employee", "hr", "benefit", "insurance", "pf",
    "gratuity", "esic", "holiday", "overtime", "shift", "allowance",
    "deduction", "tax", "tds", "grade", "band", "promotion", "transfer",
    "zyro", "dynamics", "company", "joining", "full", "final", "settlement",
    "termination", "disciplinary", "code", "ethics", "misconduct", "warning",
    "annual", "earned", "casual", "compensatory", "bereavement",
    "accommodation", "hotel", "flight", "per diem", "cab", "conveyance",
    "work", "hours", "attendance", "payroll", "increment", "rating",
}

OUT_OF_SCOPE_KEYWORDS = {
    "weather", "cricket", "football", "sports", "movie", "recipe", "cook",
    "stock", "crypto", "bitcoin", "politics", "government", "news",
    "celebrity", "music", "song", "gaming", "restaurant", "astrology",
    "horoscope", "joke", "poem", "story", "doctor", "medical diagnosis",
}

REFUSAL_MSG = (
    "I'm sorry, I can only answer HR-related questions based on Zyro Dynamics "
    "policy documents. This question appears to be outside my scope. "
    "Please contact HR directly for non-policy queries."
)

def is_hr_question(question: str):
    q_lower = question.lower()
    words   = set(re.findall(r'\b\w+\b', q_lower))
    if words & OUT_OF_SCOPE_KEYWORDS:
        return False
    if words & HR_KEYWORDS:
        return True
    return None  # ambiguous → defer to LLM

def format_docs(docs):
    seen, result = set(), []
    for doc in docs:
        c = doc.page_content.strip()
        if c not in seen:
            seen.add(c)
            result.append(c)
    return "\n\n---\n\n".join(result)

def multi_query_retrieve(question, retriever):
    try:
        raw      = (multi_query_prompt | llm | StrOutputParser()).invoke({"question": question})
        variants = [q.strip() for q in raw.strip().split("\n") if q.strip()][:3]
    except Exception:
        variants = []
    seen_ids, docs = set(), []
    for q in [question] + variants:
        try:
            for doc in retriever.invoke(q):
                doc_id = hash(doc.page_content[:100])
                if doc_id not in seen_ids:
                    seen_ids.add(doc_id)
                    docs.append(doc)
        except Exception:
            pass
    return docs or retriever.invoke(question)

# ── Main RAG chain ─────────────────────────────────────────────────────────────
def ask_bot(question: str) -> dict:
    question = question.strip()

    # Layer 1: keyword fast-path
    keyword_result = is_hr_question(question)
    if keyword_result is False:
        return {"answer": REFUSAL_MSG, "sources": []}

    # Layer 2: LLM classification for ambiguous
    if keyword_result is None:
        cls = (scope_prompt | llm | StrOutputParser()).invoke({"question": question}).strip().upper()
        if "NO" in cls and "YES" not in cls:
            return {"answer": REFUSAL_MSG, "sources": []}

    # Layer 3: multi-query retrieval
    docs = multi_query_retrieve(question, base_retriever)

    if not docs:
        return {
            "answer": "This specific information is not covered in the available HR policy documents. Please contact HR directly.",
            "sources": []
        }

    context = format_docs(docs)
    chain   = (
        {"context": RunnableLambda(lambda _: context), "question": RunnablePassthrough()}
        | rag_prompt | llm | StrOutputParser()
    )
    answer  = chain.invoke(question)
    sources = sorted(set(doc.metadata.get("source", "") for doc in docs))

    return {"answer": answer, "sources": sources}

print("✅ RAG chain with guardrails ready")

In [ ]:
# Cell 7 — Quick smoke test
test = ask_bot("What is the work from home policy at Zyro Dynamics?")
print("Q: What is the work from home policy at Zyro Dynamics?")
print("A:", test["answer"][:300], "...")

In [ ]:
# Cell 8 — Encoding helpers (base64 used by the competition)
def encode_text(text: str) -> str:
    return base64.b64encode(text.encode("utf-8")).decode("utf-8")

def decode_text(encoded: str) -> str:
    return base64.b64decode(encoded.encode("utf-8")).decode("utf-8")

print("✅ Encode/decode helpers ready")

In [ ]:
# Cell 9 — ⭐ PASTE YOUR ENCODED QUESTIONS HERE ⭐
# Replace each "PASTE_ENCODED_Q_HERE" with the actual base64 encoded question
# from your starter notebook's Cell 12.
#
# How to get them: In the starter notebook, run the cell that prints
# question_enc values, then paste them below.

QUESTIONS_ENC = {
    "Q01": "PASTE_ENCODED_Q01_HERE",
    "Q02": "PASTE_ENCODED_Q02_HERE",
    "Q03": "PASTE_ENCODED_Q03_HERE",
    "Q04": "PASTE_ENCODED_Q04_HERE",
    "Q05": "PASTE_ENCODED_Q05_HERE",
    "Q06": "PASTE_ENCODED_Q06_HERE",
    "Q07": "PASTE_ENCODED_Q07_HERE",
    "Q08": "PASTE_ENCODED_Q08_HERE",
    "Q09": "PASTE_ENCODED_Q09_HERE",
    "Q10": "PASTE_ENCODED_Q10_HERE",
    "Q11": "PASTE_ENCODED_Q11_HERE",
    "Q12": "PASTE_ENCODED_Q12_HERE",
    "Q13": "PASTE_ENCODED_Q13_HERE",
    "Q14": "PASTE_ENCODED_Q14_HERE",
    "Q15": "PASTE_ENCODED_Q15_HERE",
}

# Decode to verify
for qid, enc in QUESTIONS_ENC.items():
    if enc != f"PASTE_ENCODED_{qid}_HERE":
        print(f"{qid}: {decode_text(enc)[:80]}")
    else:
        print(f"⚠️  {qid}: not yet filled in")

In [ ]:
# Cell 10 — ⭐ ADD YOUR LINKS HERE ⭐
STREAMLIT_URL  = "PASTE_YOUR_STREAMLIT_URL_HERE"   # e.g. https://zyro-hr-bot.streamlit.app
LANGSMITH_URL  = "PASTE_YOUR_LANGSMITH_TRACE_URL_HERE"  # e.g. https://smith.langchain.com/public/xxx/r

print(f"Streamlit : {STREAMLIT_URL}")
print(f"LangSmith : {LANGSMITH_URL}")

In [ ]:
# Cell 11 — Run all 15 questions through the RAG bot & collect answers
results = []

for qid, q_enc in QUESTIONS_ENC.items():
    if q_enc.startswith("PASTE_ENCODED"):
        print(f"⚠️  Skipping {qid} — not filled in yet")
        continue

    question = decode_text(q_enc)
    print(f"\n{'='*60}")
    print(f"🔍 {qid}: {question}")

    response  = ask_bot(question)
    answer    = response["answer"]
    answer_enc = encode_text(answer)

    print(f"💬 Answer: {answer[:200]}..." if len(answer) > 200 else f"💬 Answer: {answer}")
    if response["sources"]:
        print(f"📄 Sources: {[os.path.basename(s) for s in response['sources']]}")

    results.append({
        "question_id":    qid,
        "question_enc":   q_enc,
        "answer_enc":     answer_enc,
        "streamlit_link": STREAMLIT_URL,
        "langsmith_link": LANGSMITH_URL,
    })

print(f"\n\n✅ Answered {len(results)} / 15 questions")

In [ ]:
# Cell 12 — Cell 12: Test questions (sample, from competition)
print("\n📋 SAMPLE TEST QUESTIONS (Cell 12)\n")

sample_questions = [
    "How many casual leaves are employees entitled to per year?",
    "What is the work from home policy for employees at Zyro Dynamics?",
    "What are the guidelines for business travel reimbursements?",
]

for q in sample_questions:
    print(f"Q: {q}")
    res = ask_bot(q)
    print(f"A: {res['answer']}\n")

In [ ]:
# Cell 13 — LangSmith instructions
print("""
📡 HOW TO GET YOUR LANGSMITH TRACE URL
======================================
1. Go to https://smith.langchain.com
2. Sign in with your account
3. Click on 'Projects' in the left sidebar
4. Find the project: zyro-rag-challenge
5. Click on any trace run
6. Click 'Share' button → Copy the public URL
7. Paste it in Cell 10 above as LANGSMITH_URL
8. Re-run Cell 11 to regenerate submission.csv

URL format: https://smith.langchain.com/public/xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx/r
""")

In [ ]:
# Cell 14 — Streamlit App (reference — full code in app.py repo)
print("""
🚀 STREAMLIT APP
================
The full Streamlit app is deployed at:
  https://github.com/VanjariAbhinay/zyro-hr-bot

Paste your live URL in Cell 10 above.
""")

In [ ]:
# Cell 15 (Last Cell) — Generate submission.csv
if not results:
    print("❌ No results! Fill in QUESTIONS_ENC in Cell 9 and re-run Cell 11.")
else:
    df = pd.DataFrame(results)
    # Ensure correct column order
    df = df[["question_id", "question_enc", "answer_enc", "streamlit_link", "langsmith_link"]]
    df.to_csv("submission.csv", index=False)
    print(f"✅ submission.csv generated with {len(df)} rows!")
    print("\n📊 Preview:")
    display(df[["question_id", "streamlit_link", "langsmith_link"]].head())
    print("\n⬇️  Download from File → Download → submission.csv")